In [5]:
!git clone https://github.com/zenpa1/simplefold.git
%cd simplefold
!pip install -e .
!pip install pytest

Cloning into 'simplefold'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 202 (delta 48), reused 200 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (202/202), 1.44 MiB | 7.95 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/simplefold/simplefold
Obtaining file:///content/simplefold/simplefold
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.2/118.2 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 392.5/392.5 kB 16.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  

In [5]:
!git pull origin main

From https://github.com/zenpa1/simplefold
 * branch            main       -> FETCH_HEAD
Already up to date.


In [6]:
# Execute the complete test suite with the proper python paths exposed
!PYTHONPATH="src/simplefold:src:$PYTHONPATH" pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/simplefold
configfile: pyproject.toml
plugins: hydra-core-1.3.7, langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collected 6 items                                                              

tests/test_adaln_energy.py::test_adaln_energy_forward_shape[None] PASSED [ 16%]
tests/test_adaln_energy.py::test_adaln_energy_forward_shape[free_energy_shape1] PASSED [ 33%]
tests/test_gsa.py::test_gated_sparse_attention_shape_and_blocks[128] PASSED [ 50%]
tests/test_gsa.py::test_gated_sparse_attention_shape_and_blocks[1024] PASSED [ 66%]
tests/test_integration.py::test_modified_block_and_sparse_upcycling FAILED [ 83%]
tests/test_topo_moe.py::test_topology_conditioned_moe_shape_and_auxiliary_loss PASSED [100%]

=================================== FAILURES ===================================
____

In [ ]:
#
# For licensing see accompanying LICENSE file.
# Copyright (c) 2025 Apple Inc. Licensed under MIT License.
#

# This is an example code for protein structure prediction with SimpleFold

In [ ]:
import os

os.environ["TORCH_HOME"] = r"E:\torch_cache"
os.environ["HF_HOME"] = r"E:\huggingface_cache"

In [ ]:
import sys
import numpy as np
import torch
from math import pow
import py3Dmol
from pathlib import Path
from io import StringIO
from Bio.PDB import PDBIO
from Bio.PDB import MMCIFParser, Superimposer
sys.path.append(str(Path("./src/simplefold").resolve()))

In [ ]:
# following are example amino acid sequences:
example_sequences = {
    "7ftv_A": "GASKLRAVLEKLKLSRDDISTAAGMVKGVVDHLLLRLKCDSAFRGVGLLNTGSYYEHVKISAPNEFDVMFKLEVPRIQLEEYSNTRAYYFVKFKRNPKENPLSQFLEGEILSASKMLSKFRKIIKEEINDDTDVIMKRKRGGSPAVTLLISEKISVDITLALESKSSWPASTQEGLRIQNWLSAKVRKQLRLKPFYLVPKHAEETWRLSFSHIEKEILNNHGKSKTCCENKEEKCCRKDCLKLMKYLLEQLKERFKDKKHLDKFSSYHVKTAFFHVCTQNPQDSQWDRKDLGLCFDNCVTYFLQCLRTEKLENYFIPEFNLFSSNLIDKRSKEFLTKQIEYERNNEFPVFD",
    "8cny_A": "MGPSLDFALSLLRRNIRQVQTDQGHFTMLGVRDRLAVLPRHSQPGKTIWVEHKLINILDAVELVDEQGVNLELTLVTLDTNEKFRDITKFIPENISAASDATLVINTEHMPSMFVPVGDVVQYGFLNLSGKPTHRTMMYNFPTKAGQCGGVVTSVGKVIGIHIGGNGRQGFCAGLKRSYFAS",
    "8g8r_A": "GTVNWSVEDIVKGINSNNLESQLQATQAARKLLSREKQPPIDNIIRAGLIPKFVSFLGKTDCSPIQFESAWALTNIASGTSEQTKAVVDGGAIPAFISLLASPHAHISEQAVWALGNIAGDGSAFRDLVIKHGAIDPLLALLAVPDLSTLACGYLRNLTWTLSNLCRNKNPAPPLDAVEQILPTLVRLLHHNDPEVLADSCWAISYLTDGPNERIEMVVKKGVVPQLVKLLGATELPIVTPALRAIGNIVTGTDEQTQKVIDAGALAVFPSLLTNPKTNIQKEATWTMSNITAGRQDQIQQVVNHGLVPFLVGVLSKADFKTQKEAAWAITNYTSGGTVEQIVYLVHCGIIEPLMNLLSAKDTKIIQVILDAISNIFQAAEKLGETEKLSIMIEECGGLDKIEALQRHENESVYKASLNLIEKYFS",
    "8i85_A": "MGILQANRVLLSRLLPGVEPEGLTVRHGQFHQVVIASDRVVCLPRTAAAAARLPRRAAVMRVLAGLDLGCRTPRPLCEGSLPFLVLSRVPGAPLEADALEDSKVAEVVAAQYVTLLSGLASAGADEKVRAALPAPQGRWRQFAADVRAELFPLMSDGGCRQAERELAALDSLPDITEAVVHGNLGAENVLWVRDDGLPRLSGVIDWDEVSIGDPAEDLAAIGAGYGKDFLDQVLTLGGWSDRRMATRIATIRATFALQQALSACRDGDEEELADGLTGYR",
}
seq_id = "7ftv_A"  # choose from example_sequences
aa_sequence = example_sequences[seq_id]
print(f"Predicting structure for {seq_id} with {len(aa_sequence)} amino acids.")

Predicting structure for 7ftv_A with 351 amino acids.


In [ ]:
simplefold_model = "simplefold_100M" # choose from 100M, 360M, 700M, 1.1B, 1.6B, 3B
backend = "torch" # choose from ["mlx", "torch"]
ckpt_dir = "artifacts"
output_dir = "artifacts"
prediction_dir = f"predictions_{simplefold_model}_{backend}"
output_name = f"{seq_id}"
num_steps = 500 # number of inference steps for flow-matching
tau = 0.05 # stochasticity scale
plddt = True # whether to use pLDDT confidence module
nsample_per_protein = 1 # number of samples per protein

In [ ]:
# set random seed for reproducibility
import lightning.pytorch as pl
pl.seed_everything(42, workers=True)

Seed set to 42


42

In [ ]:
from src.simplefold.wrapper import ModelWrapper, InferenceWrapper

# initialize the folding model and pLDDT model
model_wrapper = ModelWrapper(
    simplefold_model=simplefold_model,
    ckpt_dir=ckpt_dir,
    plddt=plddt,
    backend=backend,
)
device = model_wrapper.device
folding_model = model_wrapper.from_pretrained_folding_model()
plddt_model = model_wrapper.from_pretrained_plddt_model()

MLX not installed, skip importing MLX related packages.
Folding model simplefold_100M loaded with torch backend.
pLDDT output module loaded with torch backend.
pLDDT latent module loaded with torch backend.


In [ ]:
# initialize the inference module with inference configurations
inference_wrapper = InferenceWrapper(
    output_dir=output_dir,
    prediction_dir=prediction_dir,
    num_steps=num_steps,
    tau=tau,
    nsample_per_protein=nsample_per_protein,
    device=device,
    backend=backend
)

Using cache found in E:\torch_cache\hub\facebookresearch_esm_main


pLM ESM-3B loaded with torch backend.


In [ ]:
# process input sequence and run inference
batch, structure, record = inference_wrapper.process_input(aa_sequence)

batch["energy"] = torch.zeros(batch["coords"].shape[0], device=device)  # NEW — placeholder

results = inference_wrapper.run_inference(
    batch,
    folding_model,
    plddt_model,
    device=device,
)
print("NaN in output:", torch.isnan(results["sampled_coord"]).any().item())
for i, block in enumerate(folding_model.trunk.blocks):
    print(f"layer {i} aux loss:", block.mlp.last_aux_loss.item())

save_paths = inference_wrapper.save_result(
    structure,
    record,
    results,
    out_name=output_name
)

Processing input data.


100%|██████████| 1/1 [00:00<00:00,  1.20it/s]


Processing ESM features for inference...


Sampling: 100%|██████████| 500/500 [05:11<00:00,  1.61it/s]


NaN in output: False
layer 0 aux loss: 1.0464656352996826
layer 1 aux loss: 1.4314188957214355
layer 2 aux loss: 1.1922234296798706
layer 3 aux loss: 1.1825019121170044
layer 4 aux loss: 1.4591233730316162
layer 5 aux loss: 1.049290418624878
layer 6 aux loss: 1.4788687229156494
layer 7 aux loss: 1.3907074928283691


In [ ]:
# visualize the first predicted structure
pdb_path = save_paths[0]
view = py3Dmol.view(query=pdb_path)

In [ ]:
# color based on the predicted confidence
# confidence coloring from low to high: red–orange–yellow–green–blue (0 to 100)
if plddt:
    view.setStyle({'cartoon':{'colorscheme':{'prop':'b','gradient':'roygb','min':0,'max':100}}})
    view.zoomTo()
    view.show()
# color in spectrum if pLDDT is not available
else:
    view.setStyle({'cartoon':{'color':'spectrum'}})
    view.zoomTo()
    view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# visualize the all-atom structure
view.setStyle({'stick':{}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# visualize the predicted structure in 3D alongside the GT structure

def calculate_tm_score(coords1, coords2, L_target=None):
    """
    Compute TM-score for two aligned coordinate sets (numpy arrays).

    coords1, coords2: Nx3 numpy arrays (aligned atomic coordinates, e.g. CA atoms)
    L_target: length of target protein (default = len(coords1))
    """
    assert coords1.shape == coords2.shape, "Aligned coords must have same shape"
    N = coords1.shape[0]

    if L_target is None:
        L_target = N

    # distances between aligned atoms
    dists = np.linalg.norm(coords1 - coords2, axis=1)

    # scaling factor d0
    d0 = 1.24 * pow(L_target - 15, 1/3) - 1.8
    if d0 < 0.5:
        d0 = 0.5  # safeguard, as in TM-align

    # TM-score
    score = np.sum(1.0 / (1.0 + (dists/d0)**2)) / L_target
    return score

parser = MMCIFParser(QUIET=True)

# Load two structures
struct1 = parser.get_structure("ref", f"assets/{seq_id}.cif")
struct2 = parser.get_structure("prd", pdb_path)

# Select CA atoms for alignment
atoms1 = [a for a in struct1.get_atoms() if a.get_id() == 'CA']
atoms2 = [a for a in struct2.get_atoms() if a.get_id() == 'CA']
print(len(atoms1), len(atoms2))

# Superimpose
sup = Superimposer()
sup.set_atoms(atoms1, atoms2)
sup.apply(struct2.get_atoms())

# Calculate TM-score
coords1 = np.array([a.coord for a in atoms1])
coords2 = np.array([a.coord for a in atoms2])
tm_score = calculate_tm_score(coords1, coords2)

print("TM-score (0-1, higher is better): {:.3f}".format(tm_score))
print("RMSD (lower is better): {:.3f}".format(sup.rms))

# Save aligned structures to strings
io = PDBIO()

s1_buf, s2_buf = StringIO(), StringIO()
io.set_structure(struct1); io.save(s1_buf)
io.set_structure(struct2); io.save(s2_buf)

# Visualize in py3Dmol
view = py3Dmol.view(width=600, height=400)
view.addModel(s1_buf.getvalue(),"pdb")
view.addModel(s2_buf.getvalue(),"pdb")

# Color reference protein blue, predicted structure red
view.setStyle({'model': 0}, {'cartoon': {'color': 'blue'}})
view.setStyle({'model': 1}, {'cartoon': {'color': 'red'}})

# Add legend
view.addLabel("Ground Truth", {'position': {'x': 0, 'y': 0, 'z': 0}, 'backgroundColor': 'blue', 'fontColor': 'white', 'fontSize': 12})
view.addLabel("Predicted", {'position': {'x': 0, 'y': 4, 'z': 0}, 'backgroundColor': 'red', 'fontColor': 'white', 'fontSize': 12})

view.zoomTo()
view.show()

351 351
TM-score (0-1, higher is better): 0.032
RMSD (lower is better): 61.016


3Dmol.js failed to load for some reason. Please check your browser console for error messages.